# Spatial Task Visualization

This notebook visualizes the released 8-city x 8-task processed benchmark data.
It first shows the spatial block sampling design, then writes one figure per task,
with all 8 cities shown in each figure.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap
import rasterio
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent.resolve()
DATA_ROOT = ROOT / 'data'
TASKS_JSON = DATA_ROOT / 'tasks.json'
RESULT_DIR = ROOT / 'results' / 'main_eval'
FIG_DIR = ROOT / 'results' / 'spatial_tasks'
FIG_DIR.mkdir(parents=True, exist_ok=True)

CITY_ORDER = ['london','new_york','singapore','sydney','mumbai','nairobi','jakarta','cape_town']
TASK_ORDER = ['landuse','road_density','population','age_distribution','gdp','nightlight','pm25','lst_day_mean']
TASK_TITLE = {
    'landuse':'LUC',
    'road_density':'RDE',
    'population':'POP',
    'age_distribution':'AGE*',
    'gdp':'GDP',
    'nightlight':'NTL',
    'pm25':'PM25',
    'lst_day_mean':'LST',
}
SPLIT_MODEL = 'aether'
SPLIT_PROTOCOL = 'block10_5seed_mlp1024'
SPLIT_SEED = 42
SEEDS = [42, 24, 7, 0, 100]
SPLIT_KIND = 'test'
SPLIT_COLORS = {
    'empty': '#f1f5f9',
    'train': '#4c78a8',
    'val': '#f58518',
    'test': '#e45756',
}
FREQ_CMAP = {
    'test': 'Reds',
    'val': 'Oranges',
    'train': 'Blues',
}
AGE_MIDPOINTS = {
    'age_00_04': 2.0, 'age_05_14': 10.0, 'age_15_24': 20.0,
    'age_25_34': 30.0, 'age_35_44': 40.0, 'age_45_54': 50.0,
    'age_55_64': 60.0, 'age_65_74': 70.0, 'age_75_84': 80.0,
    'age_85_plus': 90.0,
}

with TASKS_JSON.open() as f:
    REGISTRY = json.load(f)['tasks']

def task_id(city, task):
    matches = [tid for tid in REGISTRY if tid.startswith(f'{city}.{task}.')]
    if not matches:
        raise KeyError((city, task))
    return matches[0]

def load_samples(city, task, max_points=18000):
    tid = task_id(city, task)
    spec = REGISTRY[tid]
    if spec.get('availability', 'full') != 'full':
        raise ValueError(f"{tid} is {spec.get('availability')} and has no public evaluation split")
    path = DATA_ROOT / spec['samples_path']
    df = pd.read_parquet(path)
    if max_points is not None and len(df) > max_points:
        df = df.sample(n=max_points, random_state=42).copy()
    return tid, spec, df

def block10_values(df, n_blocks_x=10, n_blocks_y=10):
    if {'row', 'col'}.issubset(df.columns):
        x = df['col'].to_numpy(dtype=float)
        y = df['row'].to_numpy(dtype=float)
    elif {'x', 'y'}.issubset(df.columns):
        x = df['x'].to_numpy(dtype=float)
        y = df['y'].to_numpy(dtype=float)
    else:
        raise ValueError('samples must contain row/col or x/y')
    x_span = max(float(np.nanmax(x) - np.nanmin(x)), 1e-12)
    y_span = max(float(np.nanmax(y) - np.nanmin(y)), 1e-12)
    bx = np.floor((x - float(np.nanmin(x))) / x_span * n_blocks_x).astype(int)
    by = np.floor((y - float(np.nanmin(y))) / y_span * n_blocks_y).astype(int)
    bx = np.clip(bx, 0, n_blocks_x - 1)
    by = np.clip(by, 0, n_blocks_y - 1)
    return by * n_blocks_x + bx

def fold_for_seed(df, seed=SPLIT_SEED, n_blocks_x=10, n_blocks_y=10, test_ratio=0.2, val_ratio_on_train=0.1):
    blocks = block10_values(df, n_blocks_x=n_blocks_x, n_blocks_y=n_blocks_y)
    unique_blocks = np.unique(blocks)
    train_blocks, test_blocks = train_test_split(unique_blocks, test_size=test_ratio, random_state=seed)
    train_blocks, val_blocks = train_test_split(train_blocks, test_size=val_ratio_on_train, random_state=seed)
    idx = np.arange(len(df))
    return {
        'seed': int(seed),
        'train': idx[np.isin(blocks, train_blocks)].tolist(),
        'val': idx[np.isin(blocks, val_blocks)].tolist(),
        'test': idx[np.isin(blocks, test_blocks)].tolist(),
        'meta': {
            'method': 'spatial_block',
            'n_blocks_x': int(n_blocks_x),
            'n_blocks_y': int(n_blocks_y),
            'test_ratio': float(test_ratio),
            'val_ratio_on_train': float(val_ratio_on_train),
            'train_blocks': sorted(int(x) for x in train_blocks),
            'val_blocks': sorted(int(x) for x in val_blocks),
            'test_blocks': sorted(int(x) for x in test_blocks),
        },
    }

def block_grid(df, fold):
    meta = fold['meta']
    n_blocks_x = int(meta.get('n_blocks_x', 10))
    n_blocks_y = int(meta.get('n_blocks_y', 10))
    grid = np.full((n_blocks_y, n_blocks_x), -1, dtype=np.int16)
    blocks = block10_values(df, n_blocks_x=n_blocks_x, n_blocks_y=n_blocks_y)
    grid.flat[np.unique(blocks)] = 0
    for block_id in meta.get('train_blocks', []):
        grid.flat[int(block_id)] = 1
    for block_id in meta.get('val_blocks', []):
        grid.flat[int(block_id)] = 2
    for block_id in meta.get('test_blocks', []):
        grid.flat[int(block_id)] = 3
    return grid

def plot_split(city, task, model=SPLIT_MODEL, seed=SPLIT_SEED, ax=None, title=None, show_counts=True):
    tid, spec, df = load_samples(city, task, max_points=None)
    fold = fold_for_seed(df, seed=seed)
    grid = block_grid(df, fold)
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    cmap = ListedColormap([SPLIT_COLORS['empty'], SPLIT_COLORS['train'], SPLIT_COLORS['val'], SPLIT_COLORS['test']])
    norm = mpl.colors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)
    display_grid = np.where(grid < 0, 0, grid)
    ax.imshow(display_grid, cmap=cmap, norm=norm, origin='upper')
    n_blocks_x = int(fold['meta'].get('n_blocks_x', 10))
    n_blocks_y = int(fold['meta'].get('n_blocks_y', 10))
    ax.set_xticks(np.arange(-0.5, n_blocks_x, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_blocks_y, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1.2)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.set_title(title or f'{city}\n{task}', fontsize=8)
    if show_counts:
        txt = f"blocks T/V/E: {len(fold['meta']['train_blocks'])}/{len(fold['meta']['val_blocks'])}/{len(fold['meta']['test_blocks'])}\n"
        txt += f"samples T/V/E: {len(fold['train'])}/{len(fold['val'])}/{len(fold['test'])}"
        ax.text(
            0.02, 0.02, txt, transform=ax.transAxes, fontsize=7,
            va='bottom', ha='left',
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.8, 'pad': 2},
        )
    return ax

def frequency_grid(city, task, model=SPLIT_MODEL, seeds=SEEDS, split_kind=SPLIT_KIND):
    tid, spec, df = load_samples(city, task, max_points=None)
    first_fold = fold_for_seed(df, seed=seeds[0])
    base = block_grid(df, first_fold)
    freq = np.full(base.shape, np.nan, dtype=np.float32)
    freq[base >= 0] = 0.0
    key = f'{split_kind}_blocks'
    for seed in seeds:
        fold = fold_for_seed(df, seed=seed)
        meta = fold['meta']
        n_blocks_x = int(meta.get('n_blocks_x', 10))
        for block_id in meta[key]:
            row = int(block_id) // n_blocks_x
            col = int(block_id) % n_blocks_x
            if 0 <= row < freq.shape[0] and 0 <= col < freq.shape[1] and np.isfinite(freq[row, col]):
                freq[row, col] += 1
    return freq

def plot_frequency(city, task, model=SPLIT_MODEL, seeds=SEEDS, split_kind=SPLIT_KIND, ax=None, title=None):
    freq = frequency_grid(city, task, model=model, seeds=seeds, split_kind=split_kind)
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(freq, vmin=0, vmax=len(seeds), cmap=FREQ_CMAP.get(split_kind, 'viridis'), origin='upper')
    ax.set_xticks(np.arange(-0.5, freq.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, freq.shape[0], 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1.2)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.set_title(title or f'{city}\n{task}', fontsize=8)
    return im

def color_values(df, spec, task):
    if spec['task_type'] == 'distribution':
        cols = spec['label_cols']
        mids = np.asarray([AGE_MIDPOINTS.get(c, i) for i, c in enumerate(cols)], dtype=float)
        vals = df[cols].to_numpy(dtype=float)
        vals = vals / np.clip(vals.sum(axis=1, keepdims=True), 1e-12, None)
        return vals @ mids, 'Estimated mean age', 'viridis'
    if spec['task_type'] == 'classification':
        col = spec.get('label_id_col') or 'label_id'
        return df[col].to_numpy(dtype=float), 'Class id', 'tab20'
    col = spec.get('label_col', 'label')
    vals = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
    if task in {'population', 'gdp', 'nightlight', 'road_density'}:
        vals = np.log1p(np.clip(vals, 0, None))
        return vals, f'log1p({task})', 'magma'
    return vals, task, 'viridis'

def raster_display_from_labels(spec, task):
    labels_path = spec.get('labels_path')
    if not labels_path:
        return None
    path = DATA_ROOT / labels_path
    if not path.exists():
        return None
    with rasterio.open(path) as src:
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        nodata = src.nodata
        if spec['task_type'] == 'distribution':
            data = src.read(masked=True).astype('float32')
            cols = spec['label_cols']
            mids = np.asarray([AGE_MIDPOINTS.get(c, i) for i, c in enumerate(cols)], dtype=float)
            vals = np.ma.filled(data, np.nan)
            total = np.nansum(vals, axis=0)
            weighted = np.nansum(vals * mids[:, None, None], axis=0)
            arr = weighted / np.where(total > 0, total, np.nan)
            arr[~np.isfinite(arr)] = np.nan
            return arr, extent, 'Estimated mean age', 'viridis'
        arr = src.read(1).astype('float32')
        if nodata is not None and np.isfinite(nodata):
            arr = np.where(arr == nodata, np.nan, arr)
        arr = np.where(np.isfinite(arr), arr, np.nan)
    if task in {'population', 'gdp', 'nightlight', 'road_density'}:
        arr = np.log1p(np.clip(arr, 0, None))
        return arr, extent, f'log1p({task})', 'magma'
    return arr, extent, task, 'viridis'

def raster_display_from_samples(df, spec, task, max_side=420):
    values, label, cmap = color_values(df, spec, task)
    x = pd.to_numeric(df['x'], errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(df['y'], errors='coerce').to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(values)
    x = x[mask]
    y = y[mask]
    values = values[mask]
    if x.size == 0:
        raise ValueError('no finite sample coordinates')
    xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
    ymin, ymax = float(np.nanmin(y)), float(np.nanmax(y))
    xspan = max(xmax - xmin, 1e-12)
    yspan = max(ymax - ymin, 1e-12)
    if xspan >= yspan:
        nx = int(max_side)
        ny = max(1, int(round(max_side * yspan / xspan)))
    else:
        ny = int(max_side)
        nx = max(1, int(round(max_side * xspan / yspan)))
    ix = np.floor((x - xmin) / xspan * nx).astype(int)
    iy = np.floor((ymax - y) / yspan * ny).astype(int)
    ix = np.clip(ix, 0, nx - 1)
    iy = np.clip(iy, 0, ny - 1)
    tmp = pd.DataFrame({'iy': iy, 'ix': ix, 'value': values})
    if spec['task_type'] == 'classification':
        agg = tmp.groupby(['iy', 'ix'])['value'].agg(lambda s: s.value_counts().idxmax())
    else:
        agg = tmp.groupby(['iy', 'ix'])['value'].mean()
    arr = np.full((ny, nx), np.nan, dtype='float32')
    for (row, col), val in agg.items():
        arr[int(row), int(col)] = float(val)
    return arr, [xmin, xmax, ymin, ymax], label, cmap

def raster_display(city, task):
    tid = task_id(city, task)
    spec = REGISTRY[tid]
    direct = raster_display_from_labels(spec, task)
    if direct is not None:
        return tid, spec, direct
    _, _, df = load_samples(city, task, max_points=None)
    return tid, spec, raster_display_from_samples(df, spec, task)

def plot_task_raster(ax, city, task):
    tid, spec, (arr, extent, label, cmap) = raster_display(city, task)
    if np.isfinite(arr).any() and spec['task_type'] != 'classification':
        lo, hi = np.nanpercentile(arr, [2, 98])
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            norm = None
        else:
            norm = mpl.colors.Normalize(vmin=lo, vmax=hi)
    elif spec['task_type'] == 'classification':
        finite = arr[np.isfinite(arr)]
        if finite.size:
            classes = np.arange(int(np.nanmin(finite)), int(np.nanmax(finite)) + 1)
            norm = mpl.colors.BoundaryNorm(np.r_[classes - 0.5, classes[-1] + 0.5], mpl.colormaps[cmap].N)
        else:
            norm = None
    else:
        norm = None
    im = ax.imshow(
        arr,
        extent=extent,
        origin='upper',
        cmap=cmap,
        norm=norm,
        interpolation='nearest',
        aspect='equal',
    )
    ax.set_title(city.replace('_', ' ').title(), fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_frame_on(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    return im, label

def set_equal_axis(ax, df):
    ax.set_aspect('equal', adjustable='box')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

def block10_ids(df):
    if 'block10_id' in df.columns:
        return pd.to_numeric(df['block10_id'], errors='coerce').fillna(-1).to_numpy(dtype=float)
    return block10_values(df).astype(float)

print('tasks:', len(REGISTRY))

## Spatial Block Split Design

In [ ]:
fig, axes = plt.subplots(len(CITY_ORDER), len(TASK_ORDER), figsize=(20, 18))
for i, city in enumerate(CITY_ORDER):
    for j, task in enumerate(TASK_ORDER):
        ax = axes[i, j]
        try:
            plot_split(
                city, task, SPLIT_MODEL, SPLIT_SEED, ax=ax,
                title=f"{city.replace('_', ' ').title()}\n{TASK_TITLE[task]}",
                show_counts=False,
            )
        except Exception as exc:
            ax.axis('off')
            ax.set_title(f"{city}\n{task}\n{exc}", fontsize=7)
handles = [plt.Rectangle((0, 0), 1, 1, color=SPLIT_COLORS[k]) for k in ['train', 'val', 'test', 'empty']]
fig.legend(handles, ['train', 'val', 'test', 'empty'], loc='lower center', ncol=4, frameon=False)
fig.suptitle(f'10x10 spatial-block split overview, model={SPLIT_MODEL}, seed={SPLIT_SEED}', y=0.995)
fig.tight_layout(rect=[0, 0.03, 1, 0.98])
out = FIG_DIR / f'00_all_city_task_{SPLIT_MODEL}_seed{SPLIT_SEED}_block10_split.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
print(out)
plt.show()

## Spatial Block Frequency Across Seeds

In [ ]:
fig, axes = plt.subplots(len(CITY_ORDER), len(TASK_ORDER), figsize=(20, 18))
last_im = None
for i, city in enumerate(CITY_ORDER):
    for j, task in enumerate(TASK_ORDER):
        ax = axes[i, j]
        try:
            last_im = plot_frequency(
                city, task, SPLIT_MODEL, SEEDS, SPLIT_KIND, ax=ax,
                title=f"{city.replace('_', ' ').title()}\n{TASK_TITLE[task]}",
            )
        except Exception as exc:
            ax.axis('off')
            ax.set_title(f"{city}\n{task}\n{exc}", fontsize=7)
fig.suptitle(f'10x10 block {SPLIT_KIND} frequency across 5 seeds, model={SPLIT_MODEL}', y=0.995)
fig.tight_layout(rect=[0, 0.04, 0.93, 0.98])
if last_im is not None:
    cax = fig.add_axes([0.94, 0.12, 0.015, 0.76])
    cbar = fig.colorbar(last_im, cax=cax)
    cbar.set_label(f'{SPLIT_KIND} count / {len(SEEDS)}')
out = FIG_DIR / f'00_all_city_task_{SPLIT_MODEL}_{SPLIT_KIND}_frequency_5seed_block10.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
print(out)
plt.show()

freq_rows = []
for city in CITY_ORDER:
    for task in TASK_ORDER:
        freq = frequency_grid(city, task, SPLIT_MODEL, SEEDS, SPLIT_KIND)
        vals = freq[np.isfinite(freq)]
        freq_rows.append({
            'city': city,
            'task': task,
            'split_kind': SPLIT_KIND,
            'blocks': int(vals.size),
            'min_count': int(vals.min()) if vals.size else None,
            'max_count': int(vals.max()) if vals.size else None,
            'mean_count': float(vals.mean()) if vals.size else None,
            **{f'count_{i}': int((vals == i).sum()) for i in range(len(SEEDS) + 1)},
        })
freq_table = pd.DataFrame(freq_rows)
display(freq_table)
freq_table.to_csv(FIG_DIR / f'{SPLIT_KIND}_frequency_table_{SPLIT_MODEL}_5seed_block10.csv', index=False)

## One Figure Per Task

In [ ]:
task_outputs = []
for task in TASK_ORDER:
    fig, axes = plt.subplots(2, 4, figsize=(16, 7.6), constrained_layout=True)
    for ax, city in zip(axes.ravel(), CITY_ORDER):
        try:
            im, label = plot_task_raster(ax, city, task)
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.01)
            cbar.ax.tick_params(labelsize=7)
            cbar.set_label(label, fontsize=7)
        except Exception as exc:
            ax.axis('off')
            ax.set_title(city.replace('_', ' ').title(), fontsize=10)
            ax.text(0.5, 0.5, 'Not released for evaluation', ha='center', va='center', transform=ax.transAxes, fontsize=8)
    fig.suptitle(TASK_TITLE[task], fontsize=14)
    out = FIG_DIR / f'{TASK_ORDER.index(task)+1:02d}_{task}_8cities.png'
    fig.savefig(out, dpi=240, bbox_inches='tight')
    task_outputs.append(out)
    print(out)
    plt.show()
task_outputs